# 1.2 Crowd table inspection

Evidence for the findings in the 1.2 brief of `design-tree.md`. Each section checks one finding against the crowd table built by `load_crowd_table()`.

In [1]:
import pandas as pd

from europapark_crowd.parse import load_crowd_table

table = load_crowd_table()
table["year"] = table["date"].dt.year
open_days = table[table["is_open"]]

## Rows, duplicates and missing values

March to November has 275 days, so 5 years should give 1,375 rows, each date once.

In [2]:
print("rows:", len(table), "expected:", 275 * 5)
print("duplicate dates:", table["date"].duplicated().sum())
print(table["is_open"].value_counts())
print("open days without crowd level:", open_days["crowd_level"].isna().sum())
print("open days without hours:", open_days["hours"].isna().sum())

rows: 1375 expected: 1375
duplicate dates: 0
is_open
True     1230
False     145
Name: count, dtype: int64
open days without crowd level: 0
open days without hours: 0


## Season labels per year

2019 pages show a clock icon instead of a season label. Each run below is one uninterrupted stretch of open days with the same label.

In [3]:
labels = open_days["season_label"].fillna("none")
run_id = (labels != labels.shift()).cumsum()
runs = (
    open_days.assign(label=labels, run=run_id)
    .groupby(["year", "run", "label"])["date"]
    .agg(first="min", last="max", days="count")
    .reset_index(level="run", drop=True)
)
runs

first       last  days
year label                                    
2019 none          2019-04-06 2019-11-30   220
2022 Summer Season 2022-03-26 2022-09-30   189
     Halloween     2022-10-01 2022-11-06    37
     HALLOWinter   2022-11-07 2022-11-25    19
     Winter Season 2022-11-26 2022-11-30     5
2023 Summer Season 2023-03-25 2023-09-29   189
     Halloween     2023-09-30 2023-11-05    37
     HALLOWinter   2023-11-06 2023-11-30    25
2024 Summer Season 2024-03-16 2024-09-27   191
     Halloween     2024-09-28 2024-11-03    37
     HALLOWinter   2024-11-04 2024-11-29    26
     Winter Season 2024-11-30 2024-11-30     1
2025 Summer Season 2025-03-22 2025-09-26   189
     Halloween     2025-09-27 2025-11-02    37
     HALLOWinter   2025-11-03 2025-11-28    26
     Winter Season 2025-11-29 2025-11-30     2

## Halloween phase date rule

In 2022–2025 the Halloween phase is the 37 days ending on the first Sunday of November. The same rule gives the 2019 dates.

In [4]:
def halloween_by_rule(year: int) -> tuple[pd.Timestamp, pd.Timestamp]:
    november = pd.date_range(f"{year}-11-01", periods=7)
    last = november[november.day_of_week == 6][0]
    return last - pd.Timedelta(days=36), last

halloween = open_days[open_days["season_label"] == "Halloween"].groupby("year")["date"].agg(["min", "max"])
for year in [2019, 2022, 2023, 2024, 2025]:
    first, last = halloween_by_rule(year)
    observed = [d.date().isoformat() for d in halloween.loc[year]] if year in halloween.index else "no labels"
    print(year, "rule:", first.date(), "to", last.date(), "| observed:", observed)

2019 rule: 2019-09-28 to 2019-11-03 | observed: no labels
2022 rule: 2022-10-01 to 2022-11-06 | observed: ['2022-10-01', '2022-11-06']
2023 rule: 2023-09-30 to 2023-11-05 | observed: ['2023-09-30', '2023-11-05']
2024 rule: 2024-09-28 to 2024-11-03 | observed: ['2024-09-28', '2024-11-03']
2025 rule: 2025-09-27 to 2025-11-02 | observed: ['2025-09-27', '2025-11-02']


## Closed days inside a year

From 2022 the park stays open from Halloween into winter, so no closure separates the seasons. The 2019 gap and the 2024 opening weekend show up here.

In [5]:
for year, days in table.groupby("year"):
    opened = days.loc[days["is_open"], "date"]
    inside = days[~days["is_open"] & days["date"].between(opened.min(), opened.max())]
    print(year, "first open:", opened.min().date(), "last open:", opened.max().date(),
          "closed inside:", len(inside), "days", [d.date().isoformat() for d in inside["date"]])

2019 first open: 2019-04-06 last open: 2019-11-30 closed inside: 19 days ['2019-11-04', '2019-11-05', '2019-11-06', '2019-11-07', '2019-11-08', '2019-11-09', '2019-11-10', '2019-11-11', '2019-11-12', '2019-11-13', '2019-11-14', '2019-11-15', '2019-11-16', '2019-11-17', '2019-11-18', '2019-11-19', '2019-11-20', '2019-11-21', '2019-11-22']
2022 first open: 2022-03-26 last open: 2022-11-30 closed inside: 0 days []
2023 first open: 2023-03-25 last open: 2023-11-30 closed inside: 0 days []
2024 first open: 2024-03-16 last open: 2024-11-30 closed inside: 5 days ['2024-03-18', '2024-03-19', '2024-03-20', '2024-03-21', '2024-03-22']
2025 first open: 2025-03-22 last open: 2025-11-30 closed inside: 0 days []


## Opening hours

From 2022 each label has one fixed value, so the hours look like a template. Only 2019 varies.

In [6]:
open_days.groupby([open_days["year"] == 2019, labels])["hours"].value_counts().rename_axis(["is_2019", "label", "hours"])

is_2019  label          hours      
False    HALLOWinter    11:00-19:00     96
         Halloween      09:00-18:00    148
         Summer Season  09:00-18:00    758
         Winter Season  11:00-19:00      8
True     none           09:00-20:00     83
                        09:00-18:00     67
                        09:00-19:00     60
                        11:00-19:00      5
                        11:00-20:00      3
                        09:00-23:59      1
                        09:00-21:30      1
Name: count, dtype: int64

## Crowd level per label and extremes

The crowd level tops out at 99–100. Very low values cluster in early 2022 and in HALLOWinter.

In [7]:
open_days.groupby(labels)["crowd_level"].agg(["mean", "median", "min", "max", "count"])

,mean,median,min,max,count
season_label,,,,,
HALLOWinter,26.364583,17.0,1,91,96
Halloween,70.047297,72.0,15,99,148
Summer Season,56.662269,58.0,1,100,758
Winter Season,31.875,33.0,6,64,8
none,48.940909,48.5,2,99,220


In [8]:
print("days at 99 or 100:", (open_days["crowd_level"] >= 99).sum())
open_days.loc[open_days["crowd_level"] <= 2, ["date", "crowd_level", "season_label"]]

days at 99 or 100: 23


,date,crowd_level,season_label
81,2019-05-21,2,NaN
302,2022-03-28,2,Summer Season
303,2022-03-29,2,Summer Season
305,2022-03-31,1,Summer Season
306,2022-04-01,1,Summer Season
310,2022-04-05,1,Summer Season
312,2022-04-07,2,Summer Season
313,2022-04-08,1,Summer Season
536,2022-11-17,2,HALLOWinter
537,2022-11-18,2,HALLOWinter
